# lawforge — Colab vLLM proxy

Serves `gpt-oss-20b` (MXFP4) as an OpenAI-compatible endpoint on the Colab T4/L4/A100, exposed via cloudflared tunnel.

Local machine runs the Karpathy loop and points `LAWFORGE_LLM_URL` at the tunnel URL printed below.

**Runtime**: T4 (free) or better.  
**Model**: `unsloth/gpt-oss-20b` MXFP4 ≈ 14 GB VRAM.

In [ ]:
# 1. Install deps
!pip install -q --upgrade pip
!pip install -q 'vllm>=0.6.3' 'transformers>=4.45' 'huggingface_hub>=0.25'
!pip install -q triton==3.4.0  # for MXFP4 kernels on T4
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
print('deps ok')

In [ ]:
# 2. HF login (paste your token via Colab Secrets - DO NOT hard-code)
from google.colab import userdata
from huggingface_hub import login
tok = userdata.get('HF_TOKEN')
login(token=tok)
print('hf login ok')

In [ ]:
# 3. Launch vLLM server in background
import subprocess, os, time
MODEL = 'unsloth/gpt-oss-20b'
PORT = 8000
log = open('/tmp/vllm.log', 'w')
p = subprocess.Popen([
  'python3', '-m', 'vllm.entrypoints.openai.api_server',
  '--model', MODEL,
  '--host', '0.0.0.0', '--port', str(PORT),
  '--quantization', 'mxfp4',
  '--max-model-len', '8192',
  '--dtype', 'bfloat16',
  '--gpu-memory-utilization', '0.85',
], stdout=log, stderr=subprocess.STDOUT)
print('vllm pid', p.pid)
# wait until /v1/models responds
import urllib.request, json
for i in range(180):
  try:
    r = urllib.request.urlopen(f'http://localhost:{PORT}/v1/models', timeout=3).read()
    print('vllm up:', json.loads(r)['data'][0]['id'])
    break
  except Exception as e:
    if i % 10 == 0: print(f'waiting... {i}s')
    time.sleep(1)
else:
  print('vllm did NOT start in 180s; check /tmp/vllm.log')
  !tail -50 /tmp/vllm.log

In [ ]:
# 4. Open cloudflared tunnel (free, no auth needed)
import subprocess, time, re
tun = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://localhost:8000', '--no-autoupdate'],
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
url = None
deadline = time.time() + 60
while time.time() < deadline:
  line = tun.stdout.readline()
  if not line: break
  print(line, end='')
  m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
  if m:
    url = m.group(0)
    break
print('\n\n=== Set on your local machine: ===')
print(f'export LAWFORGE_LLM_URL="{url}/v1/chat/completions"')
print(f'export LAWFORGE_LLM_MODEL="unsloth/gpt-oss-20b"')

In [ ]:
# 5. Smoke test from inside Colab
import urllib.request, json
body = json.dumps({
  'model': 'unsloth/gpt-oss-20b',
  'messages': [{'role':'user','content':'Emit a Lean 4 proof of `True`. Code only.'}],
  'max_tokens': 256, 'temperature': 0.3,
}).encode()
req = urllib.request.Request('http://localhost:8000/v1/chat/completions',
  data=body, headers={'Content-Type':'application/json', 'Authorization':'Bearer no-key'})
r = urllib.request.urlopen(req).read()
print(json.loads(r)['choices'][0]['message']['content'])

## Keep this tab open

vLLM + cloudflared run in this Colab cell. Closing the tab kills both. The free tunnel URL is ephemeral — every cell-4 rerun gets a new one.